# R3 - Behavioral summary and published reference

Select `.venv`, Restart Kernel, then Run All. This step reads the release
preprocessing log; it does not load or clean the EEG files.

Keep two overlapping cohorts separate: the 39 subjects remaining after
the paper-based a-priori exclusions, and the 35 subjects retained for ERP.
References are the published mean/SD values in Dzianok et al., Table 5,
stored in `erp.behaviour_reference`. See `reports/behaviour_summary_notes.md`.


In [ ]:
import hashlib
import json
import sys

import matplotlib.pyplot as plt
from IPython.display import display

from eegproj.config import find_root, load_config
from eegproj.evaluation.behaviour_summary import (
    load_behaviour_subjects,
    plot_behaviour_comparison,
    summarise_behaviour,
)

ROOT = find_root()
CONFIG = load_config(ROOT)
REFERENCE = CONFIG["erp"]["behaviour_reference"]
LOG_PATH = ROOT / CONFIG["paths"]["derivatives"] / "preproc_log.csv"
OUTPUT_DIR = ROOT / "results" / "erp"
FIGURE_DIR = ROOT / CONFIG["paths"]["figures"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("Input log:", LOG_PATH)
print("Paper reference:", REFERENCE["source_url"], "Table", REFERENCE["source_table"])

## 1. Check original behavioral counts and cohort membership

Rates use total stimuli and behaviorally correct trials before EEG epoch
rejection. Do not calculate hit rate from retained target epochs: that
would mix task performance with signal quality. Use each person's actual
trial counts, including recordings with counts that differ from the design.


In [ ]:
subjects, exclusions = load_behaviour_subjects(ROOT, CONFIG)
print("Before EEG QC, after a-priori exclusions:", len(subjects))
print("ERP cohort:", int(subjects["in_erp_cohort"].sum()))
display(
    subjects[
        [
            "subject",
            "in_erp_cohort",
            "hit_rate_pct",
            "target_omission_pct",
            "false_alarm_standard_pct",
            "false_alarm_distractor_pct",
            "rt_mean_ms",
        ]
    ]
)
display(exclusions)

## 2. Subject-level mean and sample SD

Average per-subject percentages and RT means with equal subject weights.
The group RT SD is the sample SD of `rt_mean_ms` across subjects; it is
not the mean of `rt_sd_ms`, which describes within-subject RT variability.
The two cohorts overlap, so this is a descriptive comparison, not a
statistical test between independent samples.


In [ ]:
summary = summarise_behaviour(subjects, CONFIG)
compact = summary[
    [
        "cohort",
        "metric",
        "n_subjects",
        "mean",
        "sd",
        "paper_mean",
        "paper_sd",
        "mean_difference_from_paper",
        "difference_unit",
    ]
]
display(compact)
comparison = summary.copy()
comparison["local_mean_sd"] = comparison.apply(lambda row: f"{row['mean']:.2f} +/- {row['sd']:.2f}", axis=1)
comparison["paper_mean_sd"] = comparison.apply(lambda row: f"{row['paper_mean']:.2f} +/- {row['paper_sd']:.2f}", axis=1)
display(
    comparison[
        [
            "cohort",
            "label",
            "n_subjects",
            "local_mean_sd",
            "paper_n_subjects",
            "paper_mean_sd",
            "mean_difference_from_paper",
            "difference_unit",
        ]
    ]
)

## 3. Save tables and provenance

Keep the pre-EEG-QC and ERP cohort labels in exported summaries. Hit rate
in the paper reference is derived from 100% minus published omission;
that derivation is marked in the table. Preserve observed differences
from published results without forcing an exact match or inventing causes.


In [ ]:
subjects.to_csv(OUTPUT_DIR / "behaviour_subjects.csv", index=False)
exclusions.to_csv(OUTPUT_DIR / "behaviour_exclusions.csv", index=False)
summary.to_csv(OUTPUT_DIR / "behaviour_summary.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "behaviour_vs_paper.csv", index=False)
manifest = {
    "pipeline_version": 2,
    "input_log": str(LOG_PATH.relative_to(ROOT)),
    "input_log_sha256": hashlib.sha256(LOG_PATH.read_bytes()).hexdigest(),
    "subjects_before_eeg_qc": subjects["subject"].tolist(),
    "subjects_erp_usable": subjects.loc[subjects["in_erp_cohort"], "subject"].tolist(),
    "n_before_eeg_qc": len(subjects),
    "n_erp_usable": int(subjects["in_erp_cohort"].sum()),
    "aggregation": "equal-weight subject means and sample SD (ddof=1)",
    "rate_denominator": "original stimulus counts before EEG epoch rejection",
    "rt_sd": "sample SD across per-subject mean RTs, not mean within-subject RT SD",
    "reference": REFERENCE,
    "comparison": "descriptive; cohorts overlap; no significance tests or exact-replication claims",
    "signal_qc_status": subjects.loc[subjects["in_erp_cohort"], "visual_review"].value_counts().to_dict(),
}
(OUTPUT_DIR / "behaviour_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved tables and manifest:", OUTPUT_DIR)

## 4. Plot subjects and published mean +/- SD

Gray dots show observed participants. Diamonds/error bars show mean and
between-subject SD. Paper individual observations are unavailable here;
only its reported mean/SD are plotted. SD bars are not confidence intervals
and can extend beyond the valid range of a rate. The two local cohorts
share subjects and must not be treated as independent groups.


In [ ]:
fig = plot_behaviour_comparison(subjects, summary, CONFIG)
for suffix in ("png", "pdf"):
    path = FIGURE_DIR / f"R3_behaviour_comparison.{suffix}"
    fig.savefig(path, dpi=CONFIG["erp"]["figure_dpi"], bbox_inches="tight")
    print("Saved:", path)
plt.show()

## Next step

Use the ERP cohort's behavioral summary when describing the final ERP
sample. Keep the pre-EEG-QC summary when comparing selection rules with
the paper. Review the flagged P300 candidates with the team and coordinate
signal QC with R2 before finalizing report results.

Remaining R3 tasks include the ERP report text, QC illustrations supplied
by R2, and F3 performance aggregation once R4/R5 provide their metric CSVs.
Generated CSV/JSON/figure outputs are saved in ignored directories.
